# 검증 데이터 분석: 안전모 미착용

두 모델의 val 결과를 확인하고 탐지 기준값과 오류 사례를 살펴봅니다. test는 사용하지 않습니다.


In [1]:
!pip -q install ultralytics==8.4.208
import json, zipfile, yaml, torch
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
drive.mount("/content/drive")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Mounted at /content/drive


In [2]:
ROOT = Path("/content/drive/MyDrive/ppe-project")
ZIP = Path("/content/drive/MyDrive/construction-ppe-5class.zip")
DATA = Path("/content/ppe_project_data/construction-ppe-5class")
assert ZIP.is_file(), f"ZIP을 찾지 못했습니다: {ZIP}"
if not (DATA / "split_manifest.csv").is_file():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        assert z.testzip() is None
        z.extractall(DATA.parent)
config = yaml.safe_load((DATA / "data.yaml").read_text())
config["path"] = str(DATA)
YAML = DATA / "data_colab.yaml"
YAML.write_text(yaml.safe_dump(config, sort_keys=False))
assert len(list((DATA / "images/val").glob("*.jpg"))) == 100
WEIGHTS = {"yolo11n": ROOT / "runs/baseline_yolo11n_seed42/weights/best.pt", "yolo26n": ROOT / "runs/proposed_yolo26n_seed42/weights/best.pt"}
for name, weight in WEIGHTS.items():
    assert weight.is_file(), f"가중치가 없습니다: {weight}"
    print(name, weight)


yolo11n /content/drive/MyDrive/ppe-project/runs/baseline_yolo11n_seed42/weights/best.pt
yolo26n /content/drive/MyDrive/ppe-project/runs/proposed_yolo26n_seed42/weights/best.pt


## 검증 결과 및 예측 이미지 저장

각 모델의 val 예측과 혼동행렬, precision-recall 곡선을 저장합니다. 기준값 0.001에서 곡선을 계산하며, 이후 예측 이미지는 기준값 0.25의 확인용 예시입니다. 미탐 여부는 정답 라벨과 비교해 확인합니다.


In [3]:
OUT = ROOT / "val_analysis"
OUT.mkdir(parents=True, exist_ok=True)
summary = {}
for name, weight in WEIGHTS.items():
    run = OUT / (name + "_metrics")
    if run.exists():
        raise FileExistsError(f"기존 결과가 있습니다: {run}")
    metrics = YOLO(str(weight)).val(data=str(YAML), split="val", imgsz=640, batch=8, device=0, conf=0.001, plots=True, project=str(OUT), name=name+"_metrics", exist_ok=False)
    box = metrics.box
    summary[name] = {"precision":float(box.mp), "recall":float(box.mr), "mAP50":float(box.map50), "mAP50_95":float(box.map), "classes":{str(metrics.names[int(c)]):{"precision":float(box.p[i]), "recall":float(box.r[i]), "AP50":float(box.ap50[i])} for i,c in enumerate(box.ap_class_index)}}
    (OUT / (name+"_summary.json")).write_text(json.dumps(summary[name],ensure_ascii=False,indent=2))
print(json.dumps(summary,ensure_ascii=False,indent=2))


Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,023 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2866.8±760.1 MB/s, size: 712.5 KB)
val: Scanning /content/ppe_project_data/construction-ppe-5class/labels/val... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 941.6it/s 0.1s
val: New cache created: /content/ppe_project_data/construction-ppe-5class/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.7it/s 4.8s
                   all        100        726      0.921      0.755      0.811      0.604
                Person        100        246      0.917      0.893      0.934      0.732
                helmet         74        146      0.964      0.767       0.82       0.64
             no_helmet         42         94      0.847      0.574      0.661      0.434
                 

## 안전모 미착용 정답이 있는 이미지 예측

val에서 `no_helmet` 정답이 있는 이미지만 고르고, 두 모델의 예측 이미지를 Drive에 저장합니다. 정답과 비교하여 미탐 여부를 판정해야 합니다. 전체 이미지 저장에는 시간이 조금 걸립니다.


In [4]:
images=[]
for label in sorted((DATA / "labels/val").glob("*.txt")):
    if any(line.split() and line.split()[0] == "2" for line in label.read_text().splitlines()):
        image=DATA / "images/val" / (label.stem+".jpg")
        if image.is_file(): images.append(str(image))
print("no_helmet 정답 포함 이미지:", len(images))
for name, weight in WEIGHTS.items():
    run=OUT / (name+"_no_helmet_examples")
    if run.exists(): raise FileExistsError(f"기존 결과가 있습니다: {run}")
    YOLO(str(weight)).predict(source=images, conf=0.25, imgsz=640, device=0, save=True, project=str(OUT), name=name+"_no_helmet_examples", exist_ok=False, verbose=False)
print("결과 폴더:", OUT)


no_helmet 정답 포함 이미지: 42
Results saved to /content/drive/MyDrive/ppe-project/val_analysis/yolo11n_no_helmet_examples
Results saved to /content/drive/MyDrive/ppe-project/val_analysis/yolo26n_no_helmet_examples
결과 폴더: /content/drive/MyDrive/ppe-project/val_analysis
